WR Grader

Cell 1

In [1]:
import pandas as pd
import numpy as np
import nflreadpy as nfl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ──────────────────────────────────────────────────────────────────
SEASONS     = list(range(2016, 2026))   # Full range including 2025
MIN_TARGETS = 50                        # Overall minimum targets
OUTPUT_DIR  = './'                      # Saves files to WRU folder

# Tier thresholds (display label only)
TIER1_TARGETS = 100
TIER2_TARGETS = 70
TIER3_TARGETS = 50

# Composite score weights (must sum to 1.0)
W_YPRR        = 0.25
W_SEPARATION  = 0.22
W_YAC         = 0.18
W_CATCH_POINT = 0.15
W_DOWN_DIST   = 0.10
W_EXPLOSIVE   = 0.07
W_SNAP_ADJ    = 0.03
# ────────────────────────────────────────────────────────────────────────────

print('✅ Config set')
print(f'   Package:      nflreadpy')
print(f'   Seasons:      {SEASONS[0]} – {SEASONS[-1]}')
print(f'   Min Targets:  {MIN_TARGETS}')
print(f'   Weights:      YPRR={W_YPRR} | Sep={W_SEPARATION} | YAC={W_YAC} | CP={W_CATCH_POINT}')
print(f'                 D&D={W_DOWN_DIST} | Exp={W_EXPLOSIVE} | Snap={W_SNAP_ADJ}')

✅ Config set
   Package:      nflreadpy
   Seasons:      2016 – 2025
   Min Targets:  50
   Weights:      YPRR=0.25 | Sep=0.22 | YAC=0.18 | CP=0.15
                 D&D=0.1 | Exp=0.07 | Snap=0.03


In [15]:
import subprocess
subprocess.run(['pip', 'install', 'nflreadpy', 'pyarrow', '--upgrade'], capture_output=True)
print('✅ nflreadpy + pyarrow installed')

Cell 2

In [4]:
print('📡 Pulling NGS receiving data (2016–2025)...')

ngs_raw = nfl.load_nextgen_stats(stat_type='receiving', seasons=SEASONS).to_pandas()

# Check available columns
print(f'   Available NGS columns: {list(ngs_raw.columns)}')

NGS_COLS = [
    'season', 'player_gsis_id', 'player_display_name', 'player_position',
    'team_abbr', 'week', 'targets', 'receptions',
    'avg_separation',
    'avg_cushion',
    'avg_intended_air_yards',
    'avg_yac',
    'avg_yac_above_expectation',
    'catch_percentage',
]

ngs_cols_available = [c for c in NGS_COLS if c in ngs_raw.columns]
ngs = ngs_raw[ngs_cols_available].copy()
ngs = ngs[ngs['player_position'] == 'WR']

ngs_seasonal = ngs[ngs['week'] == 0].copy()
ngs_weekly   = ngs[ngs['week'] > 0].copy()

print(f'\n✅ NGS seasonal rows : {len(ngs_seasonal):,}')
print(f'✅ NGS weekly rows   : {len(ngs_weekly):,}')
ngs_seasonal.head()

In [ ]:
print('📡 Pulling YPRR and snap count data (2016–2025)...')

# ── YPRR — via seasonal player stats ─────────────────────────────────────────
seasonal_raw = nfl.load_player_stats(seasons=SEASONS, summary_level='reg').to_pandas()

print(f'
   Available columns related to routes/targets:')
route_cols = [c for c in seasonal_raw.columns if any(
    k in c.lower() for k in ['route', 'yprr', 'target', 'air', 'wopr', 'racr']
)]
print(f'   {route_cols}')

yprr_cols_available = [c for c in [
    'player_id', 'player_name', 'season',
    'target_share', 'air_yards_share',
    'wopr', 'racr',
] if c in seasonal_raw.columns]

# Aggregate weekly → seasonal
yprr = seasonal_raw[yprr_cols_available].groupby(
    ['player_id', 'player_name', 'season']
).agg(
    avg_wopr            = ('wopr',             'mean'),
    avg_racr            = ('racr',             'mean'),
    avg_target_share    = ('target_share',     'mean'),
    avg_air_yards_share = ('air_yards_share',  'mean'),
).reset_index()

print(f'
✅ YPRR data pulled: {len(yprr):,} player-seasons')

# ── SNAP COUNTS ──────────────────────────────────────────────────────────────
print('
📡 Pulling snap count data...')
snaps_raw = nfl.load_snap_counts(seasons=SEASONS).to_pandas()

print(f'
   Available snap columns:')
snap_cols = [c for c in snaps_raw.columns if any(
    k in c.lower() for k in ['snap', 'player', 'season', 'week', 'position', 'route']
)]
print(f'   {snap_cols}')

snaps_available = [c for c in [
    'player', 'pfr_player_id', 'position',
    'season', 'week',
    'offense_snaps', 'offense_pct',
] if c in snaps_raw.columns]

snaps = snaps_raw[snaps_available].copy()
snaps = snaps[snaps['position'] == 'WR']

snaps_seasonal = snaps.groupby(['player', 'season']).agg(
    total_snaps  = ('offense_snaps', 'sum'),
    avg_snap_pct = ('offense_pct',   'mean'),
    weeks_played = ('week',          'count'),
).reset_index()

snaps_seasonal['avg_snap_pct'] = snaps_seasonal['avg_snap_pct'].round(3)

print(f'✅ Snap counts pulled: {len(snaps_seasonal):,} player-seasons')
print(f'
   Sample YPRR data:')
print(yprr.head(3).to_string())
print(f'
   Sample snap data:')
print(snaps_seasonal.head(3).to_string())


Cell 3

In [4]:
print('📡 Pulling nflfastR play-by-play data...')
print('⏳ This will take 5–10 minutes on first run — downloading 9 seasons of data...')

pbp_raw = nfl.load_pbp(seasons=SEASONS).to_pandas()

pbp_pass = pbp_raw[
    (pbp_raw['play_type'] == 'pass') &
    (pbp_raw['receiver_id'].notna()) &
    (pbp_raw['air_yards'].notna())
].copy()

PBP_COLS = [
    'season', 'week', 'game_id', 'play_id',
    'posteam', 'receiver_id', 'receiver', 'passer_id', 'passer',
    'pass_location', 'air_yards', 'yards_after_catch',
    'cp', 'cpoe', 'xyac_mean_yardage', 'xyac_epa', 'yac_epa',
    'complete_pass', 'incomplete_pass', 'interception',
    'yards_gained', 'down', 'ydstogo', 'yardline_100',
    'epa', 'ep', 'success',
]

pbp_cols_available = [c for c in PBP_COLS if c in pbp_pass.columns]
pbp = pbp_pass[pbp_cols_available].copy()

# Filter to WRs only
wr_ids = ngs_seasonal['player_gsis_id'].unique()
pbp = pbp[pbp['receiver_id'].isin(wr_ids)].copy()

pbp['yac_over_expected'] = pbp['yards_after_catch'] - pbp['xyac_mean_yardage']
pbp['caught'] = pbp['complete_pass'].astype(float)
pbp['air_yards_bucket'] = pd.cut(
    pbp['air_yards'],
    bins=[-20, 0, 5, 10, 15, 20, 30, 100],
    labels=['Behind LOS', '0-5', '5-10', '10-15', '15-20', '20-30', '30+']
)

print(f'✅ Targeted WR pass plays: {len(pbp):,}')
pbp.head()

In [5]:
print('🔧 Calculating QB quality adjustment factors...')

# Calculate each QB's avg CPOE per season
# This represents QB quality independent of receiver
qb_quality = pbp.groupby(['season', 'passer_id', 'passer']).agg(
    qb_targets      = ('play_id',   'count'),
    qb_avg_cpoe     = ('cpoe',      'mean'),
    qb_avg_cp       = ('cp',        'mean'),
    qb_total_epa    = ('epa',       'sum'),
).reset_index()

# Filter to QBs with meaningful sample (min 100 pass attempts)
qb_quality = qb_quality[qb_quality['qb_targets'] >= 100]

print(f'✅ QB quality factors calculated: {len(qb_quality):,} QB-seasons')

# Show top and bottom QBs by CPOE in 2025
print(f'\n   Top 5 QBs by CPOE (2025):')
print(qb_quality[qb_quality['season'] == 2025]
      .nlargest(5, 'qb_avg_cpoe')
      [['passer', 'qb_targets', 'qb_avg_cpoe']].to_string(index=False))

print(f'\n   Bottom 5 QBs by CPOE (2025):')
print(qb_quality[qb_quality['season'] == 2025]
      .nsmallest(5, 'qb_avg_cpoe')
      [['passer', 'qb_targets', 'qb_avg_cpoe']].to_string(index=False))

🔧 Calculating QB quality adjustment factors...
✅ QB quality factors calculated: 350 QB-seasons

   Top 5 QBs by CPOE (2025):
      passer  qb_targets  qb_avg_cpoe
      D.Maye         317    10.491718
   S.Darnold         317     5.643453
      J.Love         209     4.586166
    J.Burrow         128     4.460781
T.Tagovailoa         190     3.851723

   Bottom 5 QBs by CPOE (2025):
    passer  qb_targets  qb_avg_cpoe
    C.Ward         259    -8.611536
J.McCarthy         156    -7.596140
   M.Penix         138    -6.400736
  C.Stroud         271    -5.444533
B.Mayfield         301    -5.347461


In [6]:
print('🔧 Applying QB quality adjustment to play-level data...')

# Merge QB quality factors onto every play
pbp = pbp.merge(
    qb_quality[['season', 'passer_id', 'qb_avg_cpoe']],
    on=['season', 'passer_id'],
    how='left'
)

# Fill missing QB adjustments with 0 (neutral)
pbp['qb_avg_cpoe'] = pbp['qb_avg_cpoe'].fillna(0)

# Calculate receiver-adjusted CPOE
# Removes QB quality from the equation
pbp['adj_cpoe'] = pbp['cpoe'] - pbp['qb_avg_cpoe']

print(f'✅ QB adjustment applied to {len(pbp):,} plays')
print(f'\n   Raw CPOE vs Adjusted CPOE sample:')
print(pbp[['receiver', 'passer', 'season', 'cpoe', 'qb_avg_cpoe', 'adj_cpoe']]
      .dropna()
      .head(10)
      .to_string(index=False))

🔧 Applying QB quality adjustment to play-level data...
✅ QB adjustment applied to 99,185 plays

   Raw CPOE vs Adjusted CPOE sample:
  receiver   passer  season       cpoe  qb_avg_cpoe   adj_cpoe
   S.Smith J.Flacco    2016  23.208666    -0.711366  23.920033
   S.Smith J.Flacco    2016  27.201820    -0.711366  27.913187
 M.Wallace J.Flacco    2016  38.063122    -0.711366  38.774487
B.Perriman J.Flacco    2016  70.148987    -0.711366  70.860352
   K.Aiken J.Flacco    2016  43.761696    -0.711366  44.473061
B.Perriman J.Flacco    2016 -69.208687    -0.711366 -68.497322
 S.Watkins T.Taylor    2016  33.128674     0.380574  32.748100
   S.Smith J.Flacco    2016  22.489578    -0.711366  23.200945
 M.Wallace J.Flacco    2016  55.849545    -0.711366  56.560909
 S.Watkins T.Taylor    2016  21.236832     0.380574  20.856256


Cell 4

In [7]:
print('⚙️ Building play-level grading engine...')

# ── SEPARATION GRADE (via CP as coverage proxy) ───────────────────────────────
# cp = completion probability (0-1), model-based using tracking data
# High CP = receiver was open / easy throw
# Low CP = receiver was covered / difficult throw
def grade_separation(row):
    cp = row.get('cp')
    caught = row['complete_pass'] == 1
    down = row.get('down')
    ydstogo = row.get('ydstogo')

    high_leverage = (not pd.isna(down) and not pd.isna(ydstogo)
                     and down in [3, 4] and ydstogo >= 5)

    if pd.isna(cp):
        return 0.0
    if cp >= 0.75 and caught:
        return 1.0
    elif cp >= 0.75 and not caught:
        return -3.0 if high_leverage else -2.0
    elif cp >= 0.50 and caught:
        return 0.5
    elif cp >= 0.50 and not caught:
        return -1.0 if high_leverage else -0.5
    elif cp >= 0.30 and caught:
        return 1.5
    elif cp >= 0.30 and not caught:
        return -0.5
    elif cp < 0.30 and caught:
        return 2.0
    elif cp < 0.30 and not caught:
        return 0.0
    return 0.0

# ── CATCH POINT GRADE ────────────────────────────────────────────────────────
def grade_catch_point(row):
    cpoe = row.get('cpoe')
    caught = row['complete_pass'] == 1
    incomplete = row['incomplete_pass'] == 1
    down = row.get('down')
    ydstogo = row.get('ydstogo')
    
    # High leverage flag
    high_leverage = (not pd.isna(down) and not pd.isna(ydstogo) 
                     and down in [3, 4] and ydstogo >= 5)

    if pd.isna(cpoe):
        return 0.0
    if caught and cpoe > 10:
        return 1.0
    elif caught and cpoe >= 0:
        return 0.0
    elif caught and cpoe < 0:
        return 1.5
    elif incomplete and cpoe > 10:
        return -3.0 if high_leverage else -2.0  # Heavy penalty on critical downs
    elif incomplete and cpoe >= 0:
        return -2.0 if high_leverage else -1.0  # Stronger base penalty
    elif incomplete and cpoe < 0:
        return 0.0
    return 0.0

# ── YAC GRADE ────────────────────────────────────────────────────────────────
def grade_yac(row):
    yac_oe = row.get('yac_over_expected')
    caught = row['complete_pass'] == 1

    if not caught or pd.isna(yac_oe):
        return 0.0
    if yac_oe > 3.0:
        return 1.5
    elif yac_oe > 1.0:
        return 0.5
    elif yac_oe >= -1.0:
        return 0.0
    elif yac_oe >= -3.0:
        return -0.5
    else:
        return -1.0

# ── DOWN & DISTANCE MULTIPLIER ───────────────────────────────────────────────
def down_dist_multiplier(row):
    down = row.get('down')
    ydstogo = row.get('ydstogo')

    if pd.isna(down) or pd.isna(ydstogo):
        return 1.0
    if down in [3, 4] and ydstogo >= 5:
        return 1.5
    elif down in [3, 4] and ydstogo < 5:
        return 1.2
    return 1.0

# ── EXPLOSIVE PLAY BONUS ─────────────────────────────────────────────────────
def explosive_bonus(row):
    air_yards = row.get('air_yards')
    caught = row['complete_pass'] == 1

    if pd.isna(air_yards) or not caught:
        return 0.0
    if air_yards >= 30:
        return 1.0
    elif air_yards >= 20:
        return 0.5
    return 0.0

# ── APPLY GRADES TO EVERY PLAY ───────────────────────────────────────────────
print('   Grading separation (CP-based)...')
pbp['sep_grade']     = pbp.apply(grade_separation, axis=1)

print('   Grading catch point...')
pbp['cp_grade']      = pbp.apply(grade_catch_point, axis=1)

print('   Grading YAC...')
pbp['yac_grade']     = pbp.apply(grade_yac, axis=1)

print('   Applying down & distance multiplier...')
pbp['dd_multiplier'] = pbp.apply(down_dist_multiplier, axis=1)

print('   Applying explosive bonus...')
pbp['exp_bonus']     = pbp.apply(explosive_bonus, axis=1)

# ── COMPOSITE PLAY GRADE ─────────────────────────────────────────────────────
# ── COMPOSITE PLAY GRADE ─────────────────────────────────────────────────────
pbp['raw_grade'] = (
    pbp['sep_grade']  * W_SEPARATION +
    pbp['cp_grade']   * W_CATCH_POINT +
    pbp['yac_grade']  * W_YAC
)

# Apply multiplier to BOTH positive and negative grades
pbp['play_grade'] = (
    pbp['raw_grade'] * pbp['dd_multiplier'] + pbp['exp_bonus']
).round(3)

print(f'\n✅ Play grades assigned to {len(pbp):,} plays')
print(f'\n   Grade distribution:')
print(pbp['play_grade'].describe().round(3))
print(f'\n   Positive grades: {(pbp["play_grade"] > 0).sum():,}')
print(f'   Neutral grades:  {(pbp["play_grade"] == 0).sum():,}')
print(f'   Negative grades: {(pbp["play_grade"] < 0).sum():,}')
pbp[['receiver', 'season', 'air_yards', 'complete_pass',
     'sep_grade', 'cp_grade', 'yac_grade',
     'dd_multiplier', 'exp_bonus', 'play_grade']].head(10)

⚙️ Building play-level grading engine...
   Grading separation (CP-based)...
   Grading catch point...
   Grading YAC...
   Applying down & distance multiplier...
   Applying explosive bonus...

✅ Play grades assigned to 99,185 plays

   Grade distribution:
count    99185.000
mean         0.280
std          0.646
min         -1.800
25%         -0.200
50%          0.275
75%          0.650
max          3.362
Name: play_grade, dtype: float64

   Positive grades: 62,265
   Neutral grades:  3,637
   Negative grades: 33,283


,receiver,season,air_yards,complete_pass,sep_grade,cp_grade,yac_grade,dd_multiplier,exp_bonus,play_grade
0,S.Smith,2016,3.0,1.0,1.0,1.0,-1.0,1.0,0.0,0.300
1,S.Smith,2016,5.0,1.0,0.5,1.0,-0.5,1.0,0.0,0.275
2,M.Wallace,2016,16.0,1.0,0.5,1.0,-1.0,1.5,0.0,0.150
3,B.Perriman,2016,35.0,1.0,2.0,1.0,-1.0,1.0,1.0,1.700
4,K.Aiken,2016,16.0,1.0,0.5,1.0,-1.0,1.0,0.0,0.100
5,B.Perriman,2016,8.0,0.0,-0.5,0.0,0.0,1.0,0.0,-0.200
6,S.Watkins,2016,5.0,1.0,0.5,1.0,-1.0,1.2,0.0,0.120
7,S.Smith,2016,4.0,1.0,1.0,1.0,-1.0,1.0,0.0,0.300
8,M.Wallace,2016,31.0,1.0,1.5,1.0,1.5,1.2,1.0,2.650
9,S.Watkins,2016,4.0,1.0,1.0,1.0,0.0,1.0,0.0,0.650


Cell 5

In [8]:
print('🔗 Aggregating play grades to seasonal WR scores...')

# Aggregate play-level grades to seasonal totals
pbp_grades = pbp.groupby(['season', 'receiver_id', 'receiver']).agg(

    # Volume
    targets             = ('play_id',          'count'),
    receptions          = ('complete_pass',     'sum'),

    # Play grade accumulation
    total_grade         = ('play_grade',        'sum'),
    avg_grade           = ('play_grade',        'mean'),
    positive_plays      = ('play_grade',        lambda x: (x > 0).sum()),
    negative_plays      = ('play_grade',        lambda x: (x < 0).sum()),
    neutral_plays       = ('play_grade',        lambda x: (x == 0).sum()),

    # Component grades
    avg_sep_grade       = ('sep_grade',         'mean'),
    avg_cp_grade        = ('cp_grade',          'mean'),
    avg_yac_grade       = ('yac_grade',         'mean'),
    total_exp_bonus     = ('exp_bonus',         'sum'),
    high_leverage_plays = ('dd_multiplier',     lambda x: (x > 1.0).sum()),

    # Supporting stats
    avg_air_yards       = ('air_yards',         'mean'),
    avg_yac_raw         = ('yards_after_catch', 'mean'),
    avg_yac_oe          = ('yac_over_expected', 'mean'),
    avg_cpoe            = ('cpoe',              'mean'),
    total_epa           = ('epa',               'sum'),
    success_rate        = ('success',           'mean'),

).reset_index()

# Derived metrics
pbp_grades['catch_rate']         = (pbp_grades['receptions'] / pbp_grades['targets']).round(3)
pbp_grades['positive_play_rate'] = (pbp_grades['positive_plays'] / pbp_grades['targets']).round(3)
pbp_grades['negative_play_rate'] = (pbp_grades['negative_plays'] / pbp_grades['targets']).round(3)
pbp_grades['high_leverage_rate'] = (pbp_grades['high_leverage_plays'] / pbp_grades['targets']).round(3)

# Merge with NGS separation data
ngs_clean = ngs_seasonal.rename(columns={
    'player_gsis_id':       'player_id',
    'player_display_name':  'player_name',
    'player_position':      'position',
    'team_abbr':            'team',
    'targets':              'ngs_targets',      # Rename to avoid collision
    'receptions':           'ngs_receptions',
})

master = pd.merge(
    ngs_clean,
    pbp_grades,
    left_on  = ['season', 'player_id'],
    right_on = ['season', 'receiver_id'],
    how      = 'inner'
)

master['player_name'] = master['player_name'].fillna(master['receiver'])
master = master.drop(columns=['receiver', 'receiver_id', 'week'], errors='ignore')

# Filter WR only and minimum targets
master = master[master['position'] == 'WR']
master = master[master['targets'] >= MIN_TARGETS]

# Assign tiers as display label only
def assign_tier(targets):
    if targets >= TIER1_TARGETS:
        return 'Tier 1 - Featured Starter'
    elif targets >= TIER2_TARGETS:
        return 'Tier 2 - Role Starter'
    else:
        return 'Tier 3 - Rotational'

master['tier'] = master['targets'].apply(assign_tier)
master = master.sort_values(['season', 'targets'], ascending=[True, False]).reset_index(drop=True)

print(f'✅ Master dataset: {len(master):,} player-seasons')
print(f'   Seasons: {sorted(master["season"].unique())}')
print(f'   Unique WRs: {master["player_name"].nunique():,}')
print(f'\n   Tier breakdown:')
print(master['tier'].value_counts())
master.head(10)

🔗 Aggregating play grades to seasonal WR scores...
✅ Master dataset: 869 player-seasons
   Seasons: [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
   Unique WRs: 283

   Tier breakdown:
tier
Tier 1 - Featured Starter    370
Tier 2 - Role Starter        265
Tier 3 - Rotational          234
Name: count, dtype: int64


,season,player_id,player_name,position,team,ngs_targets,ngs_receptions,avg_separation,avg_cushion,avg_intended_air_yards,...,avg_yac_raw,avg_yac_oe,avg_cpoe,total_epa,success_rate,catch_rate,positive_play_rate,negative_play_rate,high_leverage_rate,tier
0,2016,00-0027150,Julian Edelman,WR,NE,159,98,2.688326,6.381912,9.488868,...,5.142857,0.266752,-1.815844,53.470497,0.517949,0.610,0.610,0.354,0.297,Tier 1 - Featured Starter
1,2016,00-0027793,Antonio Brown,WR,PIT,154,106,2.916970,5.040544,11.055714,...,4.814516,-0.147662,6.636693,53.650883,0.491803,0.678,0.678,0.279,0.295,Tier 1 - Featured Starter
2,2016,00-0031235,Odell Beckham,WR,NYG,169,101,2.758720,5.242733,11.352544,...,5.209524,0.749370,-1.461079,33.944927,0.466667,0.583,0.583,0.367,0.322,Tier 1 - Featured Starter
3,2016,00-0031408,Mike Evans,WR,TB,173,96,2.163241,5.753918,15.538266,...,1.864583,-1.945326,2.980695,62.187386,0.520231,0.555,0.555,0.324,0.329,Tier 1 - Featured Starter
4,2016,00-0030564,DeAndre Hopkins,WR,HOU,151,78,2.256239,4.325436,12.691258,...,3.494382,-0.518833,-6.247936,-2.509337,0.491124,0.527,0.527,0.408,0.325,Tier 1 - Featured Starter
5,2016,00-0026176,Jordy Nelson,WR,GB,152,97,2.668155,5.768671,13.011645,...,3.567308,-0.214158,7.245847,75.314438,0.579268,0.634,0.634,0.299,0.226,Tier 1 - Featured Starter
6,2016,00-0029608,T.Y. Hilton,WR,IND,155,91,2.596275,5.668411,13.264258,...,3.758242,-0.389450,-0.218915,60.119751,0.529032,0.587,0.587,0.381,0.271,Tier 1 - Featured Starter
7,2016,00-0027944,Julio Jones,WR,ATL,129,83,2.244711,4.956803,14.452791,...,5.009804,0.688547,7.998592,90.013290,0.633987,0.667,0.667,0.294,0.190,Tier 1 - Featured Starter
8,2016,00-0026986,Michael Crabtree,WR,LV,145,89,2.261044,5.976000,11.564207,...,2.934066,-1.443265,-0.041345,35.656536,0.493421,0.599,0.599,0.342,0.349,Tier 1 - Featured Starter
9,2016,00-0031428,Allen Robinson,WR,JAX,151,73,1.949113,4.586733,14.099470,...,2.876712,-1.138019,-6.158238,-16.236536,0.423841,0.483,0.483,0.444,0.371,Tier 1 - Featured Starter


Cell 6

In [9]:
print('📊 Building WR scores from play-level grades...')

scored = master.copy()

def normalize_within_season(df, col):
    result = df.copy()
    for season in result['season'].unique():
        mask = result['season'] == season
        values = result.loc[mask, col]
        mn, mx = values.min(), values.max()
        if mx == mn:
            result.loc[mask, col + '_score'] = 50.0
        else:
            result.loc[mask, col + '_score'] = ((values - mn) / (mx - mn) * 100).round(1)
    return result

# ── WEIGHTED AVG GRADE (sample size stabilized) ──────────────────────────────
scored['weighted_avg_grade'] = scored['avg_grade'] * np.sqrt(scored['targets'])

# ── NORMALIZE ALL METRICS ────────────────────────────────────────────────────
scored = normalize_within_season(scored, 'weighted_avg_grade')
scored = normalize_within_season(scored, 'avg_sep_grade')
scored = normalize_within_season(scored, 'avg_cp_grade')
scored = normalize_within_season(scored, 'avg_yac_grade')
scored = normalize_within_season(scored, 'avg_separation')
scored = normalize_within_season(scored, 'positive_play_rate')
scored = normalize_within_season(scored, 'high_leverage_rate')

# ── RENAME FOR CLARITY ───────────────────────────────────────────────────────
scored = scored.rename(columns={
    'weighted_avg_grade_score': 'composite_grade',
    'avg_sep_grade_score':      'separation_score',
    'avg_cp_grade_score':       'catch_point_score',
    'avg_yac_grade_score':      'yac_score',
    'avg_separation_score':     'ngs_separation_score',
    'positive_play_rate_score': 'consistency_score',
    'high_leverage_rate_score': 'leverage_score',
})

# ── RESCALE TO 40–99 ─────────────────────────────────────────────────────────
def rescale(series, new_min=40, new_max=99):
    old_min, old_max = series.min(), series.max()
    return ((series - old_min) / (old_max - old_min) * (new_max - new_min) + new_min).round(1)

scored['wr_grade'] = rescale(scored['composite_grade'])

print(f'✅ Scored {len(scored):,} player-seasons')
print(f'\n   Grade summary:')
print(scored['wr_grade'].describe().round(1))
print(f'\n   By tier:')
print(scored.groupby('tier')['wr_grade'].describe().round(1))
scored[['player_name', 'season', 'team', 'tier', 'targets',
        'avg_grade', 'separation_score', 'catch_point_score',
        'yac_score', 'consistency_score', 'wr_grade']].head(10)

📊 Building WR scores from play-level grades...
✅ Scored 869 player-seasons

   Grade summary:
count    869.0
mean      65.5
std       12.2
min       40.0
25%       57.1
50%       64.7
75%       72.8
max       99.0
Name: wr_grade, dtype: float64

   By tier:
                           count  mean   std   min   25%   50%   75%   max
tier                                                                      
Tier 1 - Featured Starter  370.0  72.6  11.5  40.0  65.0  71.6  80.2  99.0
Tier 2 - Role Starter      265.0  63.0   9.6  40.0  57.0  63.2  68.4  95.2
Tier 3 - Rotational        234.0  57.0   9.0  40.0  51.0  56.6  61.8  84.8


,player_name,season,team,tier,targets,avg_grade,separation_score,catch_point_score,yac_score,consistency_score,wr_grade
0,Julian Edelman,2016,NE,Tier 1 - Featured Starter,195,0.266333,45.6,58.7,60.8,58.7,79.3
1,Antonio Brown,2016,PIT,Tier 1 - Featured Starter,183,0.328251,76.6,77.3,27.8,77.4,87.9
2,Odell Beckham,2016,NYG,Tier 1 - Featured Starter,180,0.216972,48.4,51.3,38.6,51.2,69.7
3,Mike Evans,2016,TB,Tier 1 - Featured Starter,173,0.226260,60.8,43.5,15.3,43.5,70.4
4,DeAndre Hopkins,2016,HOU,Tier 1 - Featured Starter,169,0.125077,26.2,35.8,31.7,35.8,54.3
5,Jordy Nelson,2016,GB,Tier 1 - Featured Starter,164,0.318720,68.3,65.3,47.1,65.3,83.7
6,T.Y. Hilton,2016,IND,Tier 1 - Featured Starter,155,0.249890,43.9,52.4,34.6,52.3,72.0
7,Julio Jones,2016,ATL,Tier 1 - Featured Starter,153,0.398882,78.1,74.3,63.1,74.4,93.8
8,Michael Crabtree,2016,LV,Tier 1 - Featured Starter,152,0.171776,48.0,55.6,11.5,55.6,60.2
9,Allen Robinson,2016,JAX,Tier 1 - Featured Starter,151,0.132695,37.4,23.9,33.5,23.7,54.4


In [10]:
top_2025 = (
    scored[scored['season'] == 2025]
    .nlargest(25, 'wr_grade')
    [[
        'player_name', 'team', 'tier', 'targets',
        'avg_grade',
        'separation_score',
        'catch_point_score',
        'yac_score',
        'consistency_score',
        'wr_grade'
    ]]
    .reset_index(drop=True)
)
top_2025.index += 1
top_2025

,player_name,team,tier,targets,avg_grade,separation_score,catch_point_score,yac_score,consistency_score,wr_grade
1,Puka Nacua,LAR,Tier 1 - Featured Starter,208,0.407899,60.7,78.0,68.8,79.6,99.0
2,Jaxon Smith-Njigba,SEA,Tier 1 - Featured Starter,189,0.409354,70.2,75.0,49.5,75.2,96.4
3,Stefon Diggs,NE,Tier 1 - Featured Starter,122,0.464049,83.4,100.0,48.8,100.0,91.3
4,George Pickens,DAL,Tier 1 - Featured Starter,137,0.411182,55.2,63.9,82.3,64.0,88.1
5,Khalil Shakir,BUF,Tier 1 - Featured Starter,116,0.410940,74.5,92.6,54.1,92.6,84.1
6,Luther Burden,CHI,Tier 2 - Role Starter,74,0.512365,65.3,74.0,100.0,74.1,83.9
7,Ja'Marr Chase,CIN,Tier 1 - Featured Starter,185,0.316946,49.0,61.5,65.9,63.2,83.0
8,Marvin Mims,DEN,Tier 3 - Rotational,65,0.528077,100.0,84.3,37.2,84.5,82.4
9,Zay Flowers,BAL,Tier 1 - Featured Starter,118,0.389254,67.7,77.5,38.4,77.7,82.1
10,Deebo Samuel,WAS,Tier 2 - Role Starter,99,0.407394,69.5,77.1,63.4,77.1,80.3


In [12]:
# Pull Jefferson's individual plays in 2025
jefferson_plays = pbp[
    (pbp['season'] == 2025) &
    (pbp['receiver'].str.contains('Jefferson', na=False))
][[ 
    'receiver', 'passer', 'down', 'ydstogo',
    'air_yards', 'cp', 'cpoe', 'qb_avg_cpoe', 'adj_cpoe',
    'complete_pass', 'yards_after_catch',
    'sep_grade', 'cp_grade', 'yac_grade',
    'dd_multiplier', 'play_grade'
]].copy()

print(f'Total Jefferson targets 2025: {len(jefferson_plays)}')
print(f'Completions: {jefferson_plays["complete_pass"].sum():.0f}')
print(f'Catch rate: {jefferson_plays["complete_pass"].mean():.3f}')
print(f'\nAvg adj_cpoe: {jefferson_plays["adj_cpoe"].mean():.3f}')
print(f'Avg cp_grade: {jefferson_plays["cp_grade"].mean():.3f}')
print(f'Avg sep_grade: {jefferson_plays["sep_grade"].mean():.3f}')
print(f'\ncp_grade distribution:')
print(jefferson_plays['cp_grade'].value_counts().sort_index())
print(f'\nplay_grade distribution:')
print(jefferson_plays['play_grade'].describe().round(3))
jefferson_plays.head(15)

Total Jefferson targets 2025: 193
Completions: 113
Catch rate: 0.585

Avg adj_cpoe: 0.154
Avg cp_grade: 0.575
Avg sep_grade: 0.189

cp_grade distribution:
cp_grade
0.0     82
1.0    111
Name: count, dtype: int64

play_grade distribution:
count    193.000
mean       0.232
std        0.606
min       -0.960
25%       -0.200
50%        0.100
75%        0.650
max        2.375
Name: play_grade, dtype: float64


,receiver,passer,down,ydstogo,air_yards,cp,cpoe,qb_avg_cpoe,adj_cpoe,complete_pass,yards_after_catch,sep_grade,cp_grade,yac_grade,dd_multiplier,play_grade
90489,J.Jefferson,J.McCarthy,3.0,8.0,8.0,0.440731,-44.073086,-7.596140,-36.476944,0.0,NaN,-0.5,0.0,0.0,1.5,-0.300
90492,J.Jefferson,J.McCarthy,3.0,5.0,3.0,0.740315,25.968529,-7.596140,33.564671,1.0,1.0,0.5,1.0,-1.0,1.5,0.150
90501,J.Jefferson,J.McCarthy,3.0,8.0,6.0,0.676543,-67.654350,-7.596140,-60.058208,0.0,NaN,-1.0,0.0,0.0,1.5,-0.600
90507,J.Jefferson,J.McCarthy,1.0,10.0,17.0,0.413028,58.697216,-7.596140,66.293358,1.0,0.0,1.5,1.0,-1.0,1.0,0.500
90509,J.Jefferson,J.McCarthy,3.0,5.0,13.0,0.464892,53.510765,-7.596140,61.106907,1.0,0.0,1.5,1.0,0.0,1.5,1.275
90511,J.Jefferson,J.McCarthy,2.0,1.0,23.0,0.454289,-45.428898,-7.596140,-37.832756,0.0,NaN,-0.5,0.0,0.0,1.0,-0.200
90514,J.Jefferson,J.McCarthy,2.0,11.0,-1.0,0.859502,14.049774,-7.596140,21.645914,1.0,11.0,1.0,1.0,1.5,1.0,1.175
90693,J.Jefferson,J.McCarthy,1.0,10.0,11.0,0.697133,30.286716,-7.596140,37.882858,1.0,-2.0,0.5,1.0,-1.0,1.0,0.100
90697,J.Jefferson,J.McCarthy,1.0,2.0,2.0,0.425087,-42.508656,-7.596140,-34.912514,0.0,NaN,-0.5,0.0,0.0,1.0,-0.200
90703,J.Jefferson,J.McCarthy,2.0,13.0,30.0,0.423002,57.699806,-7.596140,65.295944,1.0,20.0,1.5,1.0,1.5,1.0,2.375
